In [ ]:
import ollama

In [ ]:
example = """<think>I see I have access to these set of tools: ['quote_get_option_fundamentals']. I have to validate which tools to call and how many tool calls are required. Let's solve it step by step.

I will begin by activating the option fundamentals tool within the fastAPI function, 'quote_get_option_fundamentals' from the function. I also need to provide the currency code for the option fundamentals. 

Since I don't have these parameters, I will proceed with the other tool names and call methods with those required parameters. 

I will use the 'quote_get_option_fundamentals' function from the QuickAPI and select either 'V' or 'MA'. 

Please note that the relevant tools are 'quote_get_option_fundamentals' and 'quote_get_options_fundamentals'. 

Let's start by connecting them to Mastercard with the correct currency code.</think>
<tool_call>[{'name': 'quote_get_option_fundamentals', 'arguments': {'symbol': 'V'}}, {'name': 'quote_get_option_fundamentals', 'arguments': {'symbol': 'MA'}}]</tool_call>"""


# example = """<think>For the given question, I have to call 'quote_get_option_fundamentals' with attribute symbol where the value would be 'V'.</think>
# <tool_call>[{'name': 'quote_get_option_fundamentals', 'arguments': {'symbol': 'V'}}]</tool_call>"""

In [ ]:
def thinking_validate(llm_gen):
    TOOL_VERIFY_PROMPT = """"You will be given a chain of thought of an LLM that tries to plan and make function/tool call. 
The step by step thinking/planning would be inside <think> </think> tags and the tool/function call would be inside <tool_call> </tool_call> tags. 
You have to validate if the step by step is reflecting the tool calls properly or not.


# Instructions:

- If the 'think' is not reflecting the actions made in inside 'tool_call': return False
- If the 'think' is not contextually idealizing the actions inside 'tool_call': return False
- If the 'think' highlighting different actions than taken inside 'tool_call': return False
- If the 'think' is indicating multiple steps but execute only one tool: return False

If all of the above instructions pass, return True


# Response Format:

Respond either True of False.
Example:
[True/False]
"""

    # def chat_with_ollama(example):
    messages = [
        # {"role": "system", "content": "You are a helpful AI assistant named SmolThink. First plan/reason/code/validate inside <think></think> tag and provide final answer to user query inside <answer></answer> tag."},
        {"role": "system", "content": TOOL_VERIFY_PROMPT},
        {"role": "user", "content": llm_gen}
    ]

    # First call: Initial chat with streaming enabled
    resp = ollama.chat(
        model="qwen3:0.6b",
        messages=messages,
        stream=False,
        think=False,
        options={
            # 'ctx': 512,
            'temperature': 0.,
            'num_predict': 2
        },   
    )

    return resp.message.content.strip().lower() == 'true'

print(thinking_validate(example))

In [ ]:
response_stream = ollama.chat(
        model="qwen3:0.6b",
        messages=[{"role": "user", "content": "Hi!"}],
        stream=False,
        think=False,
        options={
            'ctx': 512,
            'temperature': 0.,
            'num_predict': 12
        },
    )

print(response_stream.message['content'])

In [ ]:
from ddgs import DDGS

def web_search(query: str, max_results: int = 2) -> str:
    """
    Performs a DuckDuckGo web search and returns formatted markdown output.
    
    Args:
        query (str): Search query string.
        max_results (int): Number of results to return.
    
    Returns:
        str: Markdown formatted search results.
    """
    results = DDGS().text(query, max_results=max_results)
    
    output_lines = ["## Search Results"]
    
    for i, item in enumerate(results):
        title = item.get("title", "No Title")
        href = item.get("href", "")
        snippet = item.get("body", "")
        date = item.get("date", "Unknown date")
        
        output_lines.append(
            f"{i}. [{title}]({href})\n"
            f"Date published: {date}\n\n"
            f"{snippet}\n"
        )
    
    return "\n".join(output_lines)


# Example use:
if __name__ == "__main__":
    print(duck_search("Nansei Islands"))


In [ ]:
import json

with open("/Users/ohi/Documents/GitHub/NanoAgent/weights/NanoAgent-135M-grpo-hilr/train_info.json", "r") as f:
    train_info = json.load(f)

# print("Model loaded", flush=True)
# return (
#     train_info["iter_step"],
#     train_info["losses"],
#     train_info["learning_rates"],
#     train_info["all_rewards"],
#     train_info["max_rewards"],
#     train_info["std_rewards"],
#     train_info["tool_call_complexity"],
#     train_info["total_prompt_tokens"],
# )

    # train_info["iter_step"],
last = 100
all_losses = train_info["losses"]#[-last:]
learning_rates = train_info["learning_rates"]#[-last:]
all_rewards = train_info["all_rewards"]#[-last:]
max_rewards = train_info["max_rewards"]
    # train_info["std_rewards"],
# tool_call_complexity = train_info["tool_call_complexity"]#[-last:]
    # train_info["total_prompt_tokens"]
# print()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d
from scipy.interpolate import make_interp_spline
from scipy.ndimage import gaussian_filter1d

def mean_map(data, win=10):
    def _mean(x):
        while len(x) < win: x.append(0)
        return sum(x) / len(x)
    _data = []
    for i in range(len(data)):
        data_win = data[max(0, i-win+1):i+1]
        # print(i, len(data_win))
        _data.append(_mean(data_win))
    return _data



# plt.close()
fig, axes = plt.subplots(4, 1, figsize=(18, 12), dpi=600)
fig.suptitle(f"GRPO Iter: {iter}", fontsize=13)

# GRPO Loss
axes[0].plot(
    np.cumsum(all_losses) / (np.arange(len(all_losses)) + 1), color="tab:red"
)
# axes[0].plot(gaussian_filter1d(all_losses, sigma=2), color="tab:red", linewidth=2)
# axes[0].plot(mean_map(all_losses, 50), color="tab:red", alpha=0.2)

# axes[0].plot(
#     mean_map(all_losses), color="tab:red"
# )
# axes[0].plot(
#     all_losses, color="tab:red", alpha=0.1
# )
axes[0].set_title("Training Loss")
axes[0].grid(True)
# axes[0].set_ylim([-0.03, -0.05])

# all_rewards = [0 for _ in range(10)] + all_rewards

# Rewards
axes[1].plot(
    np.cumsum(all_rewards) / (np.arange(len(all_rewards)) + 1), color="tab:blue"
)
# axes[1].plot(
#     mean_map(all_rewards), color="tab:blue"
# )

# spl = make_interp_spline(all_rewards, np.arange(len(all_rewards)), k=3)
# power_smooth = spl(all_rewards)
# axes[1].plot(gaussian_filter1d(all_rewards, sigma=2.5), linewidth=2, color="tab:blue")
# axes[1].plot(mean_map(all_rewards, 50), alpha=1, color="tab:blue")
# axes[1].axhline(y=max(all_rewards), color='tab:blue', linestyle='--')
# axes[1].plot(
#     all_rewards, color="tab:blue", alpha=0.1
# )
# axes[1].plot(
#     np.cumsum(max_rewards) / (np.arange(len(max_rewards)) + 1), color="tab:orange"
# )
axes[1].set_title("All Reward (blue) | Max Reward (orange)")
axes[1].grid(True)
# axes[1].set_ylim([0, 0.2])


axes[2].plot(gaussian_filter1d(max_rewards, sigma=2.5), linewidth=2, color="tab:orange")
# axes[2].plot(max_rewards, alpha=0.6, color="tab:orange")
axes[2].plot(
    np.cumsum(max_rewards) / (np.arange(len(max_rewards)) + 1), color="tab:blue"
)
axes[2].grid(True)

# Tool call complexity
# axes[2].plot(tool_call_complexity, color="tab:purple")
# axes[2].set_title("Tool Defs")
# axes[2].grid(True)

# total_prompt_tokens
axes[3].plot(learning_rates, color="tab:orange")
axes[3].set_title("Learning Rate")
axes[3].grid(True)

# Remove [0, 1] default ticks on x-axis for bottom graph
plt.tight_layout(pad=1)
# if save_path:
#     plt.savefig(
#         f"{save_path}/plot.jpg", dpi=400, bbox_inches="tight", transparent=True
#     )
# if plot:
plt.show()

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("nvidia/Nemotron-Cascade-RL-Instruction-Following")

In [ ]:
ds['train']['instruction_id_list']

In [ ]:
ds['train']

In [ ]:
!pip install docker

In [ ]:
import docker
import os
from typing import Optional


class DockerSandbox:
    def __init__(self):
        self.client = docker.from_env()
        self.container = None

    def create_container(self):
        try:
            image, build_logs = self.client.images.build(
                path=".",
                tag="agent-sandbox",
                rm=True,
                forcerm=True,
                buildargs={},
                # decode=True
            )
        except docker.errors.BuildError as e:
            print("Build error logs:")
            for log in e.build_log:
                if 'stream' in log:
                    print(log['stream'].strip())
            raise

        # Create container with security constraints and proper logging
        self.container = self.client.containers.run(
            "agent-sandbox",
            name="python-sandbox",
            command="tail -f /dev/null",  # Keep container running
            detach=True,
            tty=True,
            mem_limit="64m",
            cpu_quota=50000,
            pids_limit=100,
            security_opt=["no-new-privileges"],
            cap_drop=["ALL"],
            environment={
                "HF_TOKEN": os.getenv("HF_TOKEN")
            },
        )

    def run_code(self, code: str) -> Optional[str]:
        if not self.container:
            self.create_container()

        # Execute code in container
        exec_result = self.container.exec_run(
            cmd=["python", "-c", code],
            user="nobody"
        )

        # Collect all output
        return exec_result.output.decode() if exec_result.output else None


    def cleanup(self):
        if self.container:
            try:
                self.container.stop()
            except docker.errors.NotFound:
                # Container already removed, this is expected
                pass
            except Exception as e:
                print(f"Error during cleanup: {e}")
            finally:
                self.container = None  # Clear the reference

# Example usage:
sandbox = DockerSandbox()

# try:
if True:
    # Define your agent code
    agent_code = """
print('Hello world')
"""

    for i in range(10):
        # Run the code in the sandbox
        output = sandbox.run_code(agent_code)
        print(output)

# finally:
#     sandbox.cleanup()

In [ ]:
sandbox.cleanup()

In [ ]:
!pip install reasoning-gym

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial

# generate a number_sorting dataset

def number_sorting(tokenizer, size):
    dataset = reasoning_gym.create_dataset(
        name="number_sorting",   # task name
        min_numbers = 3,
        max_numbers = 10,
        min_decimals = 0,
        max_decimals = 1,
        min_value = -20,
        max_value = 20,
        seed = 42,
        size = size,
        # num_fewshot=1,
        # fewshot_as_multiturn=1
    )
    score_fn = get_score_answer_fn("number_sorting")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        last_line = last_line[last_line.find('['):]
        return score_fn(last_line, entry)

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = number_sorting(tokenizer=tokenizer, size=10)

print(ds[0]['prompt'])
print('larelappa\nFinal answer' + ds[0]['answer'])
print(ds[0]['scorer']('asdasd: '+ ds[0]['answer']))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial

# generate a number_sorting dataset

def needle_haystack_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    # name = last_line.strip().split()[-1].lower()
    name = last_line.lower().strip().split()
    print('Entry:', entry)
    if entry['answer'].lower().strip() in name:
        return 1.0
    return 0
    # return score_fn(name, entry)


def needle_haystack(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="needle_haystack",   # task name
        min_num_statements = 2,
        max_num_statements = 100,
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("needle_haystack")

    print(dataset[0])

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question']}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(needle_haystack_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = needle_haystack(tokenizer=tokenizer)
idx = 102
print(ds[idx]['prompt'])
print('larelappa\n' + ds[idx]['answer'], '*')
ans = ds[idx]['scorer']('Something\nFinal Answer: ' + str(ds[idx]['answer']))
# print(ds[idx]['scorer'](ans))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial

# generate a number_sorting dataset

def syllogism(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="syllogism",   # task name
        allow_all = True,
        allow_no = True,
        allow_some = True,
        allow_some_not = True,
        invalid_ratio = 0.3,
        inversion_probability = 0.3,
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("syllogism")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        name = last_line.strip().split()[-1].capitalize()
        return score_fn(name, entry)

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = syllogism(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print('larelappa\n' + ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def alice_in_wonderland(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="aiw",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("aiw")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        digits = re.findall(r'\d+', last_line)
        if digits:
            return score_fn(digits[0], entry)
        return 0

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = alice_in_wonderland(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print('larelappa\n' + ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def propositional_logic(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="propositional_logic",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("propositional_logic")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        digits = re.findall(r'\d+', last_line)
        if digits:
            return score_fn(digits[0], entry)
        return 0

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = propositional_logic(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

In [ ]:
family_relationships
gsm_symbolic
list_functions

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def family_relationships_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))
    if len(last_line) == 0: return 0
    last_line = last_line[-1].strip().lower().split()
    answer = entry['answer'].lower().strip()
    if answer in last_line:
        p = last_line.index(answer)
        return 1 / len(last_line[p:])
    return 0

def family_relationships(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="family_relationships",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("family_relationships")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(family_relationships_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = family_relationships(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def gsm_symbolic_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    digits = re.findall(r'\d+', last_line)
    if digits:
        return score_fn(digits[0], entry)
    return 0


def gsm_symbolic(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="gsm_symbolic",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("gsm_symbolic")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(gsm_symbolic_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = gsm_symbolic(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def list_functions_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    last_line = last_line[last_line.find('['):]
    print('Fetched:', last_line[last_line.find('['):])
    return score_fn(last_line, entry)


def list_functions(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="list_functions",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("list_functions")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(list_functions_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = list_functions(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

In [ ]:
def chain_sum_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    digits = re.findall(r'[+-]?\d+', last_line)
    if digits:
        digit = float(digits[-1])
        ans = float(entry['answer'])
        if digit == ans: return 1
        return abs(digit - ans) / 5
    return 0

def chain_sum(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="chain_sum",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("chain_sum")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [
                    {'role': 'user', 'content': data['question']},
                ],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(chain_sum_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list

import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = chain_sum(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + '-2'))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def codeio_parser(llm_gen, entry, score_fn):
    # last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    # llm_gen = llm_gen.strip().replace("```", "").replace('```json', '').strip()
    # print(llm_gen)
    if '{' in llm_gen and '}' in llm_gen:
        l = llm_gen.find('{')
        r = llm_gen.find('}')
        return score_fn(llm_gen[l:r+1], entry)
    return score_fn(llm_gen[l:r+1], entry)


def codeio(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="codeio",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("codeio")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'},
                 {'role': 'assistant', 'content': '```json\n'}],
                # add_generation_prompt=True,
                tokenize=False,
                continue_final_message=True,
            ),
            'answer': data['answer'],
            'scorer': partial(codeio_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = codeio(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
ans = 'asdasdasdasdasdasdasdasdasdasdSomething\nFinal Answer: ' + "```json\n" + ds[-1]['answer'] + '\n```'
print(ans)
print(ds[-1]['scorer'](ans))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re


def acre_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip().lower()
    ans = entry['answer'].strip().lower()
    if ans in last_line.split():
        score = score_fn(last_line, entry) * 0.5
        judge_score = 0 #response_judge(entry['question'], response=llm_gen, n_tokens=JUDGE_TOKENS, ref_answer=entry['answer'])[1]
        return score + judge_score * 0.5
    return 0


def acre(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="acre",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("acre")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [
                    {'role': 'user', 'content': data['question']},
                ],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(acre_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list

import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = acre(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
ans = 'asdasdasdasdasdasdasdasdasdasdSomething\nFinal Answer: "' + ds[-1]['answer'] + 'on."'
print(ans)
print(ds[-1]['scorer'](ans))

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re
import string

def last_line_parser(inp):
    last_line = list(filter(lambda x: len(x.strip().strip(string.punctuation).strip()) > 0, inp.split('\n')))
    if last_line:
        last_line = last_line[-1].strip().lower()
        return last_line
    return ''

def word_parser(inp):
    inp = inp.lower().split()
    words = list(map(lambda x: x.strip(string.punctuation), inp))
    words = list(filter(lambda x: x.strip() != '', words))
    return words

def zebra_puzzles_parser(llm_gen, entry, score_fn):
    last_line = last_line_parser(llm_gen)
    words = word_parser(last_line)
    ans = entry['answer'].strip().lower()
    print(ans, '->', words)
    score = 0
    if ans in words:
        p = words.index(ans)
        score = (1 / len(words[p:])) * 0.5
    judge_score = 0 #response_judge(entry['question'], response=llm_gen, n_tokens=JUDGE_TOKENS, ref_answer=entry['answer'])[1]
    return min(judge_score + score, 1)


def zebra_puzzles(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="zebra_puzzles",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("zebra_puzzles")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [
                    {'role': 'user', 'content': data['question']},
                ],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(zebra_puzzles_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list

import json
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = zebra_puzzles(tokenizer=tokenizer)

phrase = '''1. Alice is directly left of the person partial to Pall Mall.
2. The person partial to Pall Mall and Bob are next to each other.
3. The person partial to Pall Mall is the person who loves the boquet of iris.
4. The person's child is named Bella is Bob.
5. The person who is the mother of Billy is the person who loves the boquet of lilies.
6. The person who loves the boquet of lilies is the Prince smoker.
7. The person who loves a carnations arrangement is the person who smokes Blue Master.
8. Arnold is the person partial to Pall Mall.
9. The person's child is named Alice is Alice.
10. The person who smokes Blue Master is in the second house.'''

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
ans = 'asdasdasdasdasdasdasdasdasdasdSomething\nFinal Answer: "' + phrase
print(ans)
print(ds[-1]['scorer'](ans))

In [ ]:
import sys
sys.path.append("../")

import json
from datasets import load_dataset
from utils.tokenizer import TOOL_TEMPLATE
from data.utils import THINK_STRINGS, tool_shuffle
import random

def TxT360_efforts_if():
    def reason_fcall_map(data):
        data = json.loads(data['messages'])
        seq = []
        for idx, turn in enumerate(data):
            if turn["role"] == "assistant":
                try:
                    think = turn['think_fast']
                    resp = turn['content']
                    seq.append({
                        'role': 'assistant',
                        'content': f"<think>\n{think.strip()}\n</think>\n{resp}"
                    })
                except:
                    return {
                        "messages": [],
                        "source": "LLM360/TxT360-3efforts/instructions-with-constraints/medium",
                    }

            elif turn['role'] == 'system':
                try:
                    assert idx == 0
                    seq.append({'role': turn['role'], 'content': turn['content'].strip() + ' ' + random.choice(THINK_STRINGS)})
                except:
                    return {
                        "messages": [],
                        "source": "LLM360/TxT360-3efforts/instructions-with-constraints/medium",
                    }

            elif turn['role'] == 'user':
                try:
                    seq.append({'role': turn['role'], 'content': turn['content']})
                except:
                    return {
                        "messages": [],
                        "source": "LLM360/TxT360-3efforts/instructions-with-constraints/medium",
                    }
        
        if seq[0]['role'] != 'system':
            seq = [{'role': 'system', 'content': 'You are a helpful AI assistant. ' + random.choice(THINK_STRINGS)}] + seq

        return {
            "messages": seq,
            "source": "LLM360/TxT360-3efforts/instructions-with-constraints/medium",
        }

    fc_dataset = load_dataset("LLM360/TxT360-3efforts", "instructions-with-constraints", split='medium')
    fc_dataset = fc_dataset.map(reason_fcall_map)
    fc_dataset = fc_dataset.filter(lambda x: len(x["messages"]) > 1)
    # fc_dataset = fc_dataset.remove_columns(["conversations"])
    return fc_dataset

In [ ]:
dat = TxT360_efforts_if()

In [ ]:
print(json.dumps(dat[12]['messages'], indent=2))

In [ ]:
from datasets import load_dataset
import sys
from datasets import concatenate_datasets
sys.path.append("../")

from utils.tokenizer import TOOL_TEMPLATE, get_tokenizer

tokenizer = get_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct", add_bos=False)

def filter_by_resp_len(ds, resp_lim:int):
    def filter_fn(data):
        for turn in data['messages']:
            if turn['role'] == 'assistant':
                if len(turn['content']) > resp_lim:
                    return False
                return True
    ds = ds.filter(filter_fn)
    return ds

def nemotron_instruct(tokenizer):
    def mapper(data):
        if len(data['tools']) > 0:
            data['messages'] = TOOL_TEMPLATE.format(tools=json.dumps(data['tools'], indent=random.randint(0, 4))) + data['messages']
        for turn in data['messages']:
            assert turn['role'] in ['system', 'user', 'assistant']
        return {
            "messages": data['messages'],
            "source": "nvidia/Nemotron-Instruction-Following-Chat-v1"
        }

    ds = load_dataset("nvidia/Nemotron-Instruction-Following-Chat-v1")
    ds = ds.remove_columns(["uuid", "license", "used_in", "reasoning", "capability_target"])
    ds = ds.map(mapper)
    ds = concatenate_datasets([ds['chat_if'], ds['structured_outputs']])
    return ds

ds = nemotron_instruct(tokenizer)
ds = filter_by_resp_len(ds, 1024*3)
print(ds)

In [ ]:
import json
tc_agent = []
with open('../data/datasets/nemotron/tool_calling.jsonl', 'r') as f:
    for l in f:
        tc_agent.append(json.loads(l))
        if len(tc_agent) > 100:
            break
print(len(tc_agent))

In [ ]:
tc_agent[0]

In [1]:
from datasets import load_dataset
import sys
import json
import random
from datasets import concatenate_datasets, Dataset
sys.path.append("../")

from utils.tokenizer import TOOL_TEMPLATE, get_tokenizer

tokenizer = get_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct", add_bos=False)

# train_ds = load_dataset(
#     "jsonl",
#     data_files="../data/datasets/nemotron/interactive_agent.jsonl",
# )
# print(train_ds)

def nemotron_agentic(tool_limit=6):
    def mapper(data):
        tools = data["tools"]
        tools = [t.get('function', t) for t in tools]
        messages = []
        called_tools = []
        
        for turn in data['messages']:
            role = turn['role']
            content = turn['content']
            if role in ['user', 'system']:
                if content.strip():
                    messages.append({'role': role, 'content': content.strip()})
            elif role == 'assistant':
                tool_calls = turn.get('tool_calls')
                if tool_calls is not None:
                    tool_calls = [t.get('function', t) for t in tool_calls]
                    called_tools.extend([t['name'] for t in tool_calls])
                    tool_calls = json.dumps(tool_calls, indent=None)
                    reasoning = turn.get('reasoning_content', '')
                    if reasoning: reasoning += "\n\n"
                    messages.append({"role": role, 'content': f"{reasoning}<tool_call>{tool_calls}</tool_call>"})
                else:
                    messages.append({'role': role, 'content': content.strip()})
            elif role == 'tool':
                messages.append({'role': 'user', 'content': f"<tool_result>{content}</tool_result>"})
            else:
                raise NotImplementedError

        # Delete excess tool defs
        # random.shuffle(tools)
        # del_idx = []
        # for i, t in enumerate(tools):
        #     if t['name'] in called_tools_name or tot_tools < tool_limit:
        #         tot_tools += 1
        #     else:
        #         del_idx.append(i)
        
        # del_idx = sorted(del_idx, reverse=True)
        # for di in del_idx:
        #     del tools[di]
        
        return {
            "messages": [{'role': 'system', 'content': TOOL_TEMPLATE.format(tools=json.dumps(tools, indent=random.choice([None, 1, 2, 3, 4, 6])))}] + messages,
            "source": "nemotron/tool_calling"
        }

    # train_ds = load_dataset(
    #     "json",
    #     data_files="datasets/nemotron/tool_calling.jsonl",
    # )
    train_ds = []
    with open('../data/datasets/nemotron/interactive_agent.jsonl', 'r') as f:
        for l in f:
            train_ds.append(json.loads(l))
            # if len(train_ds) > 100: break
    train_ds = list(map(mapper, train_ds))
    train_ds = Dataset.from_list(train_ds)
    return train_ds


ds = nemotron_agentic()
print(ds)

/Users/ohi/Documents/GitHub/NanoAgent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using SmolLM2 tokenizer
Dataset({
    features: ['messages', 'source'],
    num_rows: 19028
})


In [2]:
for turn in ds[10]['messages']:
    print("Role:", turn['role'])
    print("Content:", turn['content'])
    print("---")

Role: system
Content: You are a helpful AI assistant. You have a set of possible functions/tools inside <tools> </tools> tags. 
Based on question, you may need to make one or more function/tool calls to answer user.

You have access to the following tools/functions:
<tools>[
    {
        "name": "place_order",
        "description": "Create a new food delivery order",
        "parameters": {
            "properties": {
                "user_id": {
                    "type": "string"
                },
                "restaurants": {
                    "items": {
                        "properties": {
                            "restaurant_id": {
                                "type": "string"
                            },
                            "items": {
                                "items": {
                                    "properties": {
                                        "item_id": {
                                            "type": "string"
            

In [1]:
from datasets import load_dataset
import sys
import json
import random
from datasets import concatenate_datasets, Dataset
sys.path.append("../")

from utils.tokenizer import TOOL_TEMPLATE, get_tokenizer


def nemotron_agentic(tool_limit=6):
    def mapper(data):
        tools = data["tools"]
        tools = [t.get('function', t) for t in tools]
        messages = []
        called_tools = []
        
        for turn in data['messages']:
            role = turn['role']
            content = turn['content']
            if role in ['user', 'system']:
                if content.strip():
                    messages.append({'role': role, 'content': content.strip()})
            elif role == 'assistant':
                tool_calls = turn.get('tool_calls')
                if tool_calls is not None:
                    tool_calls = [t.get('function', t) for t in tool_calls]
                    called_tools.extend([t['name'] for t in tool_calls])
                    tool_calls = json.dumps(tool_calls, indent=None)
                    reasoning = turn.get('reasoning_content', '')
                    if reasoning: reasoning += "\n\n"
                    messages.append({"role": role, 'content': f"{reasoning}<tool_call>{tool_calls}</tool_call>"})
                else:
                    messages.append({'role': role, 'content': content.strip()})
            elif role == 'tool':
                messages.append({'role': 'user', 'content': f"<tool_result>{content}</tool_result>"})
            else:
                raise NotImplementedError

        # Delete excess tool defs
        # random.shuffle(tools)
        # del_idx = []
        # for i, t in enumerate(tools):
        #     if t['name'] in called_tools_name or tot_tools < tool_limit:
        #         tot_tools += 1
        #     else:
        #         del_idx.append(i)
        
        # del_idx = sorted(del_idx, reverse=True)
        # for di in del_idx:
        #     del tools[di]
        
        return {
            "messages": [{'role': 'system', 'content': TOOL_TEMPLATE.format(tools=json.dumps(tools, indent=random.choice([None, 1, 2, 3, 4, 6])))}] + messages,
            "source": "nemotron/interactive_agent"
        }

    train_ds = []
    with open('../data/datasets/nemotron/interactive_agent.jsonl', 'r') as f:
        for l in f:
            train_ds.append(json.loads(l))
    train_ds = list(map(mapper, train_ds))
    train_ds = Dataset.from_list(train_ds)
    return train_ds


# ds = nemotron_agentic()

/Users/ohi/Documents/GitHub/NanoAgent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import sys
sys.path.append("../")
from utils.tokenizer import get_tokenizer

tokenizer = get_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct")
print(tokenizer.eos_token, tokenizer.eos_token_id)

Using SmolLM2 tokenizer
|||| 38733


In [12]:
tokenizer.chat_template = """{% for message in messages %}
{% if loop.first and messages[0]['role'] != 'system' %}
{{ '\n# system:\n\nYou are a helpful AI assistant.||||' }}
{% endif %}
{{'\n# ' + message['role'] + ':\n\n' + message['content'] + '||||'}}
{% endfor %}
{% if add_generation_prompt %}
{{ '\n# assistant:' }}
{% endif %}"""

In [13]:
msgs = [{"role": "user", "content": "hi. how are you"}]
# print("$", end="")
# print(tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True), end="$")
toks = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True)

print("$", end="")
for t in toks:
    print(f"{tokenizer.decode(t)}({t})", end="")
print("$", end="")

$
(198)#(19) system(817):(42)
(198)
(198)You(2683) are(359) a(253) helpful(5356) AI(5646) assistant(11173).||(13004)||(1331)
(198)
(198)#(19) user(2914):(42)
(198)
(198)hi(6004).(30) how(638) are(359) you(346)||||(38733)
(198)
(198)#(19) assistant(11173):(42)
(198)$

In [10]:
# print(tokenizer.encode("||||\n"))
print("$"+tokenizer.decode(1116)+"$")

$

$


In [ ]:
print(tokenizer.encode("\n# system"))
print(tokenizer.encode("# assistant"))
print(tokenizer.encode("# tool"))
print("*", end="")
print(tokenizer.decode(16836))
print("*")
print(tokenizer.encode("\n\n---"))

[19, 817]
[19, 11173]
[19, 1763]
* ----------
*
[198, 198, 5583]


In [ ]:


content = {"messages":[[{"content":"I\u2019m planning a simple dinner and want the dish to follow a pattern: the main ingredient and the cooking method should start with the same letter, just like \u201cSalmon saut\u00e9\u201d (salmon, saut\u00e9) or \u201cBeef braise\u201d (beef, braise). Could you suggest a tasty, single\u2011step recipe that fits this rule?","role":"user"},{"content":"Chicken chargrill  \nMain ingredient: chicken breast (or thigh)  \nCooking method: chargrill (high\u2011heat grill or grill pan)  \n\n**Ingredients**  \n- 1\u202flb (\u2248\u202f450\u202fg) boneless, skinless chicken breast or thighs  \n- 2\u202ftbsp olive oil  \n- 1\u202ftsp smoked paprika  \n- \u00bd\u202ftsp garlic powder  \n- Salt and freshly ground black pepper to taste  \n- Optional: squeeze of lemon juice for serving  \n\n**Single\u2011step preparation**  \n1. Toss the chicken pieces in olive oil, smoked paprika, garlic powder, salt, and pepper, then place them directly on a pre\u2011heated grill or grill pan set to high heat. Cook, turning once, until the exterior is nicely charred and the interior reaches 165\u202f\u00b0F (74\u202f\u00b0C), about\u202f4\u20115\u202fminutes per side.  \n\nServe immediately, optionally drizzled with a little lemon juice.","role":"assistant"},{"content":"Could you give me another single\u2011step recipe that follows the same\u2011letter pattern (main ingredient and cooking method start with the same letter), but this time include a placeholder for the main ingredient so I can swap it out later? For example, you might write something like \u201c[protein] \u2026\u201d in the ingredient list. Please keep the recipe concise, single\u2011step, and make sure at least one square\u2011bracket placeholder appears in your response.","role":"user"},{"content":"[Protein] poach  \nIngredients:  \n- 1\u202flb [Protein] (e.g., salmon fillet, chicken breast)  \n- 4\u202fcups water  \n- 1\u202ftsp salt  \n- Optional aromatics: bay leaf, peppercorns, lemon slice  \n\nSingle\u2011step preparation:  \nPlace the [Protein] in a saucepan with water, salt, and any aromatics; bring to a gentle simmer and poach for 8\u201110\u202fminutes (until the protein is opaque and reaches a safe internal temperature), then remove and serve.","role":"assistant"}],[{"content":"You are a health advice AI, providing users with information on healthy living and nutrition.","role":"system"},{"content":"Can you suggest some good text-to-speech apps for me?","role":"user"},{"content":"Sure, here are some highly rated text-to-speech apps that you might find useful:\n\n1. Google Text-to-Speech: This is a screen reader application developed by Google for its Android operating system. It powers applications to read aloud the text on the screen which support many languages.\n\n2. Narrator\u2019s Voice: This app offers a wide range of options, including language selection, voice selection, and speech speed.\n\n3. Speech Central: Available on iOS and Windows, Speech Central offers a variety of features including web page reader, eBook and document reader, RSS news reader, and more.\n\n4. Voice Dream Reader: This is a highly customizable reader that lets you control voice speed, pitch, volume, pronunciation, and more. It supports all major eBook formats and also integrates with Dropbox and Google Drive.\n\n5. Natural Reader: This software turns text into spoken words. You can listen to webpages, PDFs, and other documents with this app. It is available for both Windows and Mac.\n\n6. TextAloud 4: This is a Windows app that uses voice synthesis to convert text into spoken audio. It's best for those who need to read a lot of text.\n\n7. Pocket: More than just a text-to-speech app, Pocket is actually a service that helps you manage a reading list of articles from the Internet. It has a text-to-speech feature that reads out loud saved articles.\n\nPlease note that availability may vary depending on your device and location. Make sure to check each app's features and see which one suits your needs best.","role":"assistant"}],[{"content":"Could you write a short, sweet poem about the moment you pull a fresh pan de guayaba from the oven? Please make sure the answer includes a title wrapped in double angular brackets, like <<your title>>.","role":"user"},{"content":"<<Warm Ember Whisper>>\n\nFrom the oven\u2019s sigh, a golden glow,  \nPan de guayaba, soft as sunrise glow.  \nSteam curls like whispers, sweet perfume,  \nGuava\u2019s heart beating in warm, tender bloom.  \n\nA gentle lift\u2014heat kisses the air,  \nSugar and sun mingle, tender care.  \nOne bite, a hush, the world pauses sweet,  \nMorning\u2019s love, fresh from the heat.","role":"assistant"}],[{"content":"Could you give me a brief, two\u2011paragraph guide on how to handle a disagreement between crew members on a construction site? Please make sure the response contains no words written in all capital letters and that it is exactly two paragraphs, separated by a blank line.","role":"user"},{"content":"When a disagreement arises, first pause the work and ask the involved crew members to step aside from the task at hand. Give each person a chance to explain their perspective without interruption, and take notes to capture the key points. Summarize what you heard back to them to confirm understanding, and look for any miscommunication or assumptions that may have sparked the conflict. If emotions are running high, suggest a short break before moving on to the next step.\n\nAfter the facts are clear, involve the site supervisor or a designated mediator to help find a practical solution that respects safety and productivity. Encourage the parties to propose compromises and focus on shared goals, such as meeting deadlines and maintaining quality. Document the agreed\u2011upon actions, assign responsibility for any follow\u2011up tasks, and schedule a brief check\u2011in later in the day to ensure the resolution is working and that no new tensions have emerged.","role":"assistant"},{"content":"That was a solid guide \u2014 thanks! Could you now put together a quick, three\u2011step checklist that a crew leader can keep on the site wall to remind everyone how to de\u2011escalate a dispute? Please make sure the checklist items are concise, each on its own line, and that the entire response is in English with no mention of the word \u201ctoe.\u201d","role":"user"},{"content":"pause work and listen calmly  \nrepeat each point, seek common ground  \nagree on next steps and note the decision","role":"assistant"}],[{"content":"hey i need some advice on fixing a stubborn conveyor belt issue weve got at the warehouse and when you reply can you skip using periods in your response like this instead of ending sentences with dots use other ways to separate thoughts maybe commas or breaks i do that too sometimes when im texting quickly","role":"user"},{"content":"hey i hear you stubborn conveyor belt issues can really throw off the whole workflow so lets troubleshoot this step by step first thing id check is the alignment because even a slight misalignment can cause the belt to drift or snag over time make sure the rollers and pulleys are parallel and clean debris buildup often sneaks up on you and creates uneven movement  \n\nnext take a look at the tension if its too loose the belt might slip if its too tight it could strain the motor or wear out components faster adjusting tension evenly across both sides usually helps  \n\noh and the drive motor and gearbox are worth a glance listen for unusual noises grinding or whining sounds often point to bearing issues or lubrication needs sometimes the problem isnt the belt at all but the motor struggling to keep consistent torque  \n\nalso consider the load pattern are packages always placed on one side that kind of imbalance over time messes with tracking and wear patterns try to encourage more centered loading if possible  \n\nlastly check the idler rollers spin freely sometimes one seizes up and drags creating resistance you might not notice it at first but it adds up  \n\nif none of that narrows it down maybe it\u2019s time to inspect the belt itself for cracks tears or delamination especially if its been in use a while  \n\nlet me know what you\u2019ve already tried and we can dig deeper together","role":"assistant"}],[{"content":"Please write a brief, vivid description of a sunrise over a quiet lake, making sure the response consists of exactly two paragraphs separated by a blank line. The narrative should flow naturally and conclude with the single word \u201cswim\u201d.","role":"user"},{"content":"The first light tiptoes across the horizon, spilling amber and rose across the still surface of the lake. Mist curls lazily from the water\u2019s edge, catching the new colors and turning them into fleeting ribbons of gold. The distant trees stand as dark silhouettes, their reflections trembling ever so slightly as the sun lifts higher, painting the sky with promises of a day yet to unfold.\n\nA gentle hush settles over the scene, broken only by the soft lapping of water against the shore, inviting the world to linger in this quiet marvel. As the sky brightens, the water shimmers like a field of molten glass, urging any soul nearby to step in and become part of the sunrise\u2019s gentle embrace, to feel the cool embrace of the dawn and simply swim","role":"assistant"},{"content":"Please revise the sunrise description you just gave so that it includes a title formatted as\u202f<<...>> (double angular brackets) at the very beginning, followed by the two\u2011paragraph narrative exactly as before. Keep the same vivid imagery, maintain the two\u2011paragraph structure separated by a blank line, and ensure the piece still ends with the single word \u201cswim\u201d. Follow the title formatting constraint precisely.","role":"user"},{"content":"<<Morning Mirror>>\nThe first light tiptoes across the horizon, spilling amber and rose across the still surface of the lake. Mist curls lazily from the water\u2019s edge, catching the new colors and turning them into fleeting ribbons of gold. The distant trees stand as dark silhouettes, their reflections trembling ever so slightly as the sun lifts higher, painting the sky with promises of a day yet to unfold.\n\nA gentle hush settles over the scene, broken only by the soft lapping of water against the shore, inviting the world to linger in this quiet marvel. As the sky brightens, the water shimmers like a field of molten glass, urging any soul nearby to step in and become part of the sunrise\u2019s gentle embrace, to feel the cool embrace of the dawn and simply swim","role":"assistant"},{"content":"Please rewrite the sunrise piece one more time, this time giving it a fresh title of your choosing enclosed in double angular brackets (<<\u2026>>) placed at the very start. Keep the vivid imagery and the two\u2011paragraph structure exactly as before, with the paragraphs separated by a single blank line (i.e., two line breaks) and the whole work concluding with the single word \u201cswim\u201d. Be sure to honor the paragraph\u2011count constraint so that only two paragraphs appear.","role":"user"},{"content":"<<Dawn on Glass>>\nThe first blush of morning spills gold and rose across the lake\u2019s glassy surface, each ripple catching fire as the sun lifts its head. Mist rises in lazy tendrils, wrapping the shoreline in a soft veil that shimmers with the new light. Silhouetted pines stand as dark sentinels, their reflections trembling ever so slightly, while the sky unfurls a tapestry of pastel promise.\n\nA hush settles, broken only by the gentle lapping of water against the shore, inviting the world to linger in this quiet marvel. As the horizon brightens, the lake glows like molten amber, coaxing any soul to step in, become part of the sunrise\u2019s embrace, and simply swim","role":"assistant"}]],"source":["nvidia\/Nemotron-Instruction-Following-Chat-v1","HuggingFaceTB\/smoltalk\/systemchats-30k","nvidia\/Nemotron-Instruction-Following-Chat-v1","nvidia\/Nemotron-Instruction-Following-Chat-v1","nvidia\/Nemotron-Instruction-Following-Chat-v1","nvidia\/Nemotron-Instruction-Following-Chat-v1"],"ctx_len":2636}

print(tokenizer.apply_chat_template(content['messages'][0], tokenize=False))

<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:4: SyntaxWarning: invalid escape sequence '\/'
/var/folders/zh/bn7bm48j5vqfv6z0x11fq0hh0000gn/T/ipykernel_35317/4246372271.py:4: SyntaxWarning: invalid escape sequence '\/'
  content = {"messages":[[{"content":"I\u2019m planning a simple dinner and want the dish to follow a pattern: the main ingredient and the cooking method should start with the same letter, just like \u201cSalmon saut\u00e9\u201d (salmon, saut\u00e9) or \u201cBeef braise\u20

Using SmolLM2 tokenizer
|||| 38733
# SYSTEM:

You are a helpful AI assistant.||||

# USER:

I’m planning a simple dinner and want the dish to follow a pattern: the main ingredient and the cooking method should start with the same letter, just like “Salmon sauté” (salmon, sauté) or “Beef braise” (beef, braise). Could you suggest a tasty, single‑step recipe that fits this rule?||||

# ASSISTANT:

Chicken chargrill  
Main ingredient: chicken breast (or thigh)  
Cooking method: chargrill (high‑heat grill or grill pan)  

**Ingredients**  
- 1 lb (≈ 450 g) boneless, skinless chicken breast or thighs  
- 2 tbsp olive oil  
- 1 tsp smoked paprika  
- ½ tsp garlic powder  
- Salt and freshly ground black pepper to taste  
- Optional: squeeze of lemon juice for serving  

**Single‑step preparation**  
1. Toss the chicken pieces in olive oil, smoked paprika, garlic powder, salt, and pepper, then place them directly on a pre‑heated grill or grill pan set to high heat. Cook, turning once, until th

In [9]:
ds['messages'][110]

[{'content': 'You are a helpful AI assistant. You have a set of possible functions/tools inside <tools> </tools> tags. \nBased on question, you may need to make one or more function/tool calls to answer user.\n\nYou have access to the following tools/functions:\n<tools>[\n  {\n    "name": "place_order",\n    "description": "Create a new food delivery order",\n    "parameters": {\n      "properties": {\n        "user_id": {\n          "type": "string"\n        },\n        "restaurants": {\n          "items": {\n            "properties": {\n              "restaurant_id": {\n                "type": "string"\n              },\n              "items": {\n                "items": {\n                  "properties": {\n                    "item_id": {\n                      "type": "string"\n                    },\n                    "customizations": {\n                      "type": "string"\n                    }\n                  },\n                  "required": [\n                    "it

In [6]:
from data.utils import (
    THINK_STRINGS,
    code_markdown_filter,
    extract_tag,
    filter_by_resp_len,
    filter_non_english,
    filter_python_package,
    pack_data,
    remove_code_mentions,
    short_code,
    tool_shuffle,
)

In [7]:
ds2 = filter_by_resp_len(ds, 1024)

Filter: 100%|██████████| 19028/19028 [00:01<00:00, 11817.64 examples/s]


In [8]:
ds2

Dataset({
    features: ['messages', 'source'],
    num_rows: 9185
})